# Baseline de riesgo crediticio sin variables posteriores a la decisión

Este notebook implementa las secciones 9–11 de `proposal.md`. La predicción se sitúa después de recibir la solicitud y consultar el historial crediticio, pero antes de la aprobación, la tasa y el desembolso. Usa solo préstamos históricamente aprobados cuyo estado final es `Fully Paid` o `Charged Off`, incluidos los otorgados con la política de crédito anterior ("Does not meet the credit policy"); las métricas no representan automáticamente a las solicitudes rechazadas ni a los préstamos todavía `Current`.

Datos: se cargan con la base común de [`src/data.py`](../src/data.py), que define la población, el target y la partición para todo el equipo (ver [`data/Readme.md`](../data/Readme.md)). Las nueve variables son una selección provisional para un baseline numérico, no una selección óptima demostrada.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path.cwd().parent))
from src.data import SEED, cargar_prestamos, particionar, separar_X_y

FEATURES = (
    'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal',
    'total_acc', 'fico_range_low', 'delinq_2yrs', 'inq_last_6mths',
)

## Población y variables

`cargar_prestamos` devuelve solo los préstamos con resultado definitivo. `Charged Off` es la clase positiva (1) y `Fully Paid` la negativa (0); los préstamos "Does not meet the credit policy" se etiquetan según su estado final. De todas sus columnas, el baseline usa las nueve de `FEATURES`. Los faltantes se conservan hasta ajustar la imputación dentro del pipeline.

In [2]:
prestamos = cargar_prestamos()
X, y = separar_X_y(prestamos, FEATURES)

print(f'Préstamos con resultado definitivo: {len(y):,}')
print(f'Charged Off: {int(y.sum()):,} ({y.mean():.2%})')
print('Porcentaje de nulos por predictor:')
print((X.isna().mean() * 100).round(3).to_string())
del X, y

Préstamos con resultado definitivo: 1,348,059
Charged Off: 269,320 (19.98%)
Porcentaje de nulos por predictor:
annual_inc        0.000
dti               0.028
open_acc          0.002
pub_rec           0.002
revol_bal         0.000
total_acc         0.002
fico_range_low    0.000
delinq_2yrs       0.002
inq_last_6mths    0.002


## Entrenamiento, validación y prueba

La partición viene fijada en la columna `particion` de la base común: 20 % para prueba y el 80 % restante dividido en entrenamiento y validación (80/20), dando 64/16/20 en total. Ambas divisiones son estratificadas y usan `SEED = 42`. La prueba no participa en la elección de variables, transformaciones ni umbral.

In [3]:
entrenamiento, validacion, prueba = particionar(prestamos)
X_entrenamiento, y_entrenamiento = separar_X_y(entrenamiento, FEATURES)
X_validacion, y_validacion = separar_X_y(validacion, FEATURES)
X_prueba, y_prueba = separar_X_y(prueba, FEATURES)
del prestamos, entrenamiento, validacion, prueba

for nombre, etiquetas in (
    ('Entrenamiento', y_entrenamiento),
    ('Validación', y_validacion),
    ('Prueba', y_prueba),
):
    print(f'{nombre}: {len(etiquetas):,} préstamos; Charged Off {etiquetas.mean():.2%}')

Entrenamiento: 862,757 préstamos; Charged Off 19.98%
Validación: 215,690 préstamos; Charged Off 19.98%
Prueba: 269,612 préstamos; Charged Off 19.98%


## Regresión logística

La mediana, el escalado y los coeficientes se ajustan solo con entrenamiento. No se ponderan clases en este primer baseline: el efecto de otras configuraciones se comparará después en validación.

In [4]:
baseline = Pipeline([
    ('imputacion', SimpleImputer(strategy='median')),
    ('escalado', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=1000, random_state=SEED)),
])
baseline.fit(X_entrenamiento, y_entrenamiento)
prob_validacion = baseline.predict_proba(X_validacion)[:, 1]
print(f'Iteraciones del modelo: {baseline.named_steps["modelo"].n_iter_[0]}')

Iteraciones del modelo: 9


## Umbral de decisión

Como no se definió un costo de falsos positivos frente a falsos negativos, se escoge de forma provisional el umbral que maximiza F1 en validación. Es una referencia técnica, no una política para aprobar préstamos. El umbral queda fijo antes de evaluar prueba.

In [5]:
precision_curva, recall_curva, umbrales = precision_recall_curve(
    y_validacion, prob_validacion,
)
denominador = precision_curva[:-1] + recall_curva[:-1]
f1_curva = np.divide(
    2 * precision_curva[:-1] * recall_curva[:-1], denominador,
    out=np.zeros_like(denominador), where=denominador > 0,
)
indice_mejor = int(np.argmax(f1_curva))
umbral = float(umbrales[indice_mejor])
print(f'Umbral elegido en validación: {umbral:.4f}; F1: {f1_curva[indice_mejor]:.4f}')

Umbral elegido en validación: 0.1831; F1: 0.3734


## Evaluación

Aquí PR-AUC se calcula como *average precision* (AP). Se muestran métricas de validación y de prueba con el mismo umbral. Los resultados del notebook de exploración no son directamente comparables porque usan otras variables y otra partición.

In [6]:
def calcular_metricas(etiquetas, probabilidades, umbral_fijo):
    predicciones = (probabilidades >= umbral_fijo).astype('int8')
    return {
        'ROC-AUC': roc_auc_score(etiquetas, probabilidades),
        'PR-AUC (AP)': average_precision_score(etiquetas, probabilidades),
        'recall': recall_score(etiquetas, predicciones),
        'precisión': precision_score(etiquetas, predicciones, zero_division=0),
        'F1': f1_score(etiquetas, predicciones),
    }

prob_prueba = baseline.predict_proba(X_prueba)[:, 1]
resultados = pd.DataFrame(
    [calcular_metricas(y_validacion, prob_validacion, umbral),
     calcular_metricas(y_prueba, prob_prueba, umbral)],
    index=['Validación', 'Prueba'],
)
print(resultados.round(4).to_string())
tn, fp, fn, tp = confusion_matrix(
    y_prueba, (prob_prueba >= umbral).astype('int8'),
).ravel()
print(f'Prueba: TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}')

            ROC-AUC  PR-AUC (AP)  recall  precisión      F1
Validación   0.6336       0.2855  0.7337     0.2504  0.3734
Prueba       0.6331       0.2848  0.7324     0.2498  0.3725
Prueba: TN=97,255, FP=118,493, FN=14,416, TP=39,448


## Alcance de estos resultados

Esta es una evaluación aleatoria estratificada entre préstamos aprobados y resueltos, no una validación prospectiva. La comprobación temporal de la sección 10 de `proposal.md` queda pendiente porque los préstamos recientes aún no tienen suficientes resultados definitivos. Tampoco se ha demostrado que estas nueve variables sean las mejores.